CALIBRATE THE WATERMARK

In [0]:
%sql
--mesurer le retard des événements /
SELECT min(payload:event_time::timestamp) AS min_et,
       max(payload:event_time::timestamp) AS max_et,
       max(timestampdiff(HOUR, payload:event_time::timestamp, ingested_at)) AS retard_max_h
FROM bank.bronze.transactions_raw;

In [0]:
%sql

--vérification des invariants 
SELECT
  (SELECT count(*) FROM bank.bronze.transactions_raw_p)           AS bronze,
  (SELECT count(*) FROM bank.silver.transactions_valid)           AS valid,
  (SELECT count(*) FROM bank.silver.transactions_quarantine)      AS quarantine,
  (SELECT count(DISTINCT txn_id) FROM bank.silver.transactions_valid) AS valid_distinct,
  (SELECT count(*) FROM bank.silver.transactions)                 AS silver_dedup,
  (SELECT sum(n_tx) FROM bank.gold.daily_account_totals)          AS gold_n_tx;

In [0]:
%sql
--lis les expectations dans l’event log

SELECT e.dataset, e.name,
       sum(e.passed_records) AS passed, sum(e.failed_records) AS failed
FROM (
  SELECT explode(from_json(details:flow_progress:data_quality:expectations,
         'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS e
  FROM event_log(TABLE(bank.silver.transactions_valid))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress:data_quality IS NOT NULL)
GROUP BY ALL;

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath("..")) # racine du dépôt bank-realtime-lakehouse
from generators.tx_generator import generate

In [0]:
generate(n=200) #invalid_ratio = 0.02

Flow 'bank.silver.transactions_valid' failed to meet the expectation. Violated expectations: 'amount_positive'. Input data: '{"tx_typed":{"txn_id":"963b1d48-cc97-4016-9ccd-30ba28737c65","card_id":"C00078","pan":"4318393352904228","account_id":"A00039","amount":-132.99,"currency":"EUR","merchant_name":"Fnac","merchant_mcc":5732,"country":"FR","channel":"contactless","event_time":"2026-11-17T10:26:23.658Z","device_id":null,"source_file":"/Volumes/bank/ops/landing/transactions/tx_20261117T102848658596_000.json","ingested_at":"2026-11-17T10:31:31.108Z"}}'. Output record: '{"txn_id":"963b1d48-cc97-4016-9ccd-30ba28737c65","card_id":"C00078","pan":"4318393352904228","account_id":"A00039","amount":-132.99,"currency":"EUR","merchant_name":"Fnac","merchant_mcc":5732,"country":"FR","channel":"contactless","event_time":"2026-11-17T10:26:23.658Z","device_id":null,"source_file":"/Volumes/bank/ops/landing/transactions/tx_20261117T102848658596_000.json","ingested_at":"2026-11-17T10:31:31.108Z"}'. Missing input data: false

In [0]:
import json, uuid, datetime as dt

def inject_tx(delay_hours, amount=999.99):
    ref = spark.sql("""
      SELECT to_json(struct(key, topic, partition, timestamp)) AS env, value,
             payload:event_time::timestamp AS et
      FROM bank.bronze.transactions_raw_p
      WHERE payload:txn_id IS NOT NULL AND payload:amount::double > 0
      ORDER BY et DESC LIMIT 1""").first()
    msg = json.loads(ref["value"])
    new_et = ref["et"] - dt.timedelta(hours=delay_hours)
    msg.update(txn_id=f"LATE-{uuid.uuid4().hex[:8]}", amount=amount,
               event_time=new_et.strftime("%Y-%m-%dT%H:%M:%SZ"))
    env = json.loads(ref["env"]) | {"value": json.dumps(msg), "offset": 10**9}
    dbutils.fs.put(f"/Volumes/bank/ops/landing/transactions/manual_{msg['txn_id']}.json",
                   json.dumps(env), overwrite=True)
    return msg["txn_id"], msg["account_id"], new_et.date()

#print(inject_tx(24))       # 1 jour de retard : dans le watermark
#print(inject_tx(24 * 5))   # 5 jours de retard : hors watermark

In [0]:
print(inject_tx(1))       # 1 jour de retard : dans le watermark
print(inject_tx(72)) 

In [0]:
%sql
SELECT distinct channel FROM bank.silver.transactions;


SELECT * FROM bank.gold.daily_account_totals
WHERE account_id ="A00100"

ORDER BY day

In [0]:
%sql
SELECT 'bronze_p' AS couche, payload:txn_id::string AS txn_id, payload:event_time::timestamp AS et
FROM bank.bronze.transactions_raw_p WHERE payload:txn_id::string LIKE 'LATE-%'
UNION ALL
SELECT 'valid', txn_id, event_time FROM bank.silver.transactions_valid WHERE txn_id LIKE 'LATE-%'
UNION ALL
SELECT 'silver_dedup', txn_id, event_time FROM bank.silver.transactions WHERE txn_id LIKE 'LATE-%';

--SELECT max(event_time) FROM bank.silver.transactions;   -- le "max déjà vu" du stream

In [0]:
%sql
SELECT max(event_time) FROM bank.silver.transactions;   -- le "max déjà vu" du stream

In [0]:
%sql
DESCRIBE HISTORY bank.silver.transactions;

In [0]:
%sql
SELECT percentile(timestampdiff(MINUTE, payload:event_time::timestamp, ingested_at),
                  array(0.5, 0.99, 0.999)) AS retard_min_p50_p99_p999,
       max(timestampdiff(MINUTE, payload:event_time::timestamp, ingested_at)) AS retard_max
FROM bank.bronze.transactions_raw_p;